# Hari 8 Sesi f — Jaringan Rantai Pasok Oktalingual (40 Tahap, Notebook `ipynb`)

> **Catatan Duplikasi (Langkah 0)**: Topik bursa/order-book/market-making telah ada di `day002h` dan `day004g`, sehingga sesi `day008f` memakai topik cadangan resmi: **jaringan rantai pasok 3 pabrik / 4 gudang / 10 toko dengan kebijakan `(s,S)`, peramalan EWMA, pengiriman, keterlambatan, kit 4-SKU, dan biaya**.

Notebook ini mengimplementasikan seluruh alur **40 tahap (`s01`–`s40`)** secara mandiri pada beban notebook (`5 hari x 4.000 event/hari`) serta beban paritas (`3 hari x 8.000 event/hari`), menulis ke `../projects/c/day008f_supply_chain_octalingual/out/ipynb/`, dan memverifikasi paritas bit-identik lintas 8 bahasa.

In [1]:
import json, pathlib, sys, time
PROJ = pathlib.Path('../projects/c/day008f_supply_chain_octalingual')
sys.path.insert(0, str(PROJ / 'py'))
import analysis, clearing, engine, entities, inventory, rng, svg
OUT = PROJ / 'out' / 'ipynb'
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
cfg = entities.load_config(str(PROJ / 'config.kv'))
print('Konfigurasi dimuat: master_seed =', cfg['master_seed'], ', num_days =', cfg['num_days'])

Konfigurasi dimuat: master_seed = 20261126 , num_days = 5


## Tahap 1–9 (`s01`–`s09`): Selftest RNG & Alokasi, Konfigurasi, Entitas, Skrip Kebijakan/Lelang, dan Pemanasan

In [2]:
t0 = time.perf_counter()
analysis.write_s01_to_s09(cfg, str(OUT))
t_s01_09 = time.perf_counter() - t0
print((OUT / 's01_selftest.txt').read_text().strip())

ok,h_rng,h_alloc,inv_ok
1,3df66645d51ba189,01a513d19e593c4a,1


## Tahap 10 (`s10`): Simulasi Utama Multi-Hari (5 Hari x 4.000 Event)

In [3]:
t0 = time.perf_counter()
sim = entities.Sim(cfg, True, str(OUT), 4000)
engine.sim_run_days(sim, cfg['num_days'], 4000, 0, -1, 1)
sim.close_logs()
t_s10 = time.perf_counter() - t0
print('Total terjual per SKU:', sim.total_sold, '| Total kit D:', sim.total_kits, '| Pelanggaran invarian:', sim.inv_violations)

Total terjual per SKU: [24542, 23338, 22293, 11050] | Total kit D: 5687 | Pelanggaran invarian: 0


## Tahap 11–38 (`s11`–`s38`): Audit Invarian, Kliring, Kit, Sapuan, Replay, Paritas, Kompresi, & Replay Buku Besar

In [4]:
t0 = time.perf_counter()
analysis.write_s11_to_s38(sim, cfg, str(OUT))
t_s11_38 = time.perf_counter() - t0
print('s11:', (OUT / 's11_invariants.csv').read_text().splitlines()[1])
print('s33:', (OUT / 's33_replay.csv').read_text().splitlines()[1])
print('s34:', (OUT / 's34_parity.csv').read_text().splitlines()[1])
print('s37:', (OUT / 's37_compress.csv').read_text().splitlines()[1])
print('s38:', (OUT / 's38_ledger_replay.csv').read_text().splitlines()[1])

s11: 0,1
s33: bdb3b74cb6f7600e,bdb3b74cb6f7600e,6efb4dc349be6c5a,6efb4dc349be6c5a,1
s34: 3,8000,bdb3b74cb6f7600e,6efb4dc349be6c5a,103974,8893,0
s37: 4096,7618,0.5377,1
s38: 0,0,1


## Tahap 39–40 (`s39`–`s40`): 10 Grafik SVG Mandiri, Manifest Berkas, dan Paritas Lintas 8 Bahasa

In [5]:
t0 = time.perf_counter()
svg.write_s39_svgs(sim, str(OUT))
t_s39 = time.perf_counter() - t0
tot = t_s01_09 + t_s10 + t_s11_38 + t_s39
(OUT / 's40_summary.json').write_text(json.dumps({'lang':'ipynb','days':5,'ev_per_day':4000,'ev_total':cfg['num_days']*4000,'toolchain':'python3.14 + nbclient','total_sec':round(tot, 3)}) + '\n')
mf = ['file,lines,bytes,fnv1a']
for p in sorted(OUT.rglob('*')):
    if p.is_dir() or p.name == 'files_manifest.csv': continue
    rel = p.relative_to(OUT).as_posix(); b = p.read_bytes(); nl = b.count(b'\n')
    mf.append(f'{rel},{nl},{len(b)},{rng.fnv1a_bytes(b):016x}')
(OUT / 'files_manifest.csv').write_text('\n'.join(mf) + '\n')
langs = ['c','cpp','rust','go','ruby','py','ts','ipynb']
pfiles = ['s01_selftest.txt','s04_policy_script.csv','s05_alloc_script.csv','s06_schedule_head.csv','s34_parity.csv']
for pf in pfiles:
    ref = (PROJ / 'out' / 'c' / pf).read_text()
    for lg in langs:
        assert (PROJ / 'out' / lg / pf).read_text() == ref, f'{pf} mismatch in {lg}'
print('Jumlah berkas ipynb:', len(mf)-1, '| Paritas 8 bahasa (S01/S04/S05/S06/S34): 100% BIT-IDENTIK!')

Jumlah berkas ipynb: 104 | Paritas 8 bahasa (S01/S04/S05/S06/S34): 100% BIT-IDENTIK!
